# Reworking DnD classes

First, let's handle imports.

In [1]:
from enum import Enum
from pydantic import BaseModel, Field, computed_field
from typing import Annotated, Dict, List, Literal, Union

Now we can define our static rules through Enums.

In [2]:
class Ability(str, Enum):
	STR = "Strength"
	DEX = "Dexterity"
	CON = "Constitution"
	INT = "Intelligence"
	WIS = "Wisdom"
	CHA = "Charisma"


class Size(str, Enum):
	SMALL = "Small"
	MEDIUM = "Medium"

In [3]:
class AbilityScores(BaseModel):
	strength: int = Field(get=1, le=30, default=10)
	dexterity: int = Field(get=1, le=30, default=10)
	constitution: int = Field(get=1, le=30, default=10)
	intelligence: int = Field(get=1, le=30, default=10)
	wisdom: int = Field(get=1, le=30, default=10)
	charisma: int = Field(get=1, le=30, default=10)

/var/folders/kc/1lldfptj3kdc2v4flyhnrxr80000gn/T/ipykernel_76162/2007603616.py:2: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'get'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  strength: int = Field(get=1, le=30, default=10)
/var/folders/kc/1lldfptj3kdc2v4flyhnrxr80000gn/T/ipykernel_76162/2007603616.py:3: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'get'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  dexterity: int = Field(get=1, le=30, default=10)
/var/folders/kc/1lldfptj3kdc2v4flyhnrxr80000gn/T/ipykernel_76162/2007603616.py:4: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecat

Let's make a base model for races now and then get into classes.

In [4]:
class BaseRace(BaseModel):
	size: Size = Size.MEDIUM
	speed: int = 30
	languages: List[str] = ["Common"]

In [ ]:
class Dwarf(BaseRace):
	race_type: Literal["Dwarf"] = "Dwarf"
	darkvision_radius: int = 120


class Elf(BaseRace):
	race_type: Literal["Elf"] = "Elf"
	darkvision_radius: int = 60
	fey_ancestry: bool = True
	keen_senses: List[str] = ["Insight", "Perception", "Survival"]
	trance: bool = True


class Dragonborn(BaseRace):
	race_type: Literal["Dragonborn"] = "Dragonborn"
	draconic_ancestry: str
	breath_weapon_damage_str: str

In [6]:
DnDRace = Annotated[Union[Elf, Dragonborn], Field(discriminator="race_type")]

Now let's go over base classes.

In [7]:
class BaseClassLevel(BaseModel):
  level: int = Field(ge=1, le=20, default=1)
  subclass: str | None = None

In [8]:
class Fighter(BaseClassLevel):
  class_type: Literal["Fighter"] = "Fighter"
  fighting_style: str

  @computed_field
  def action_surges(self) -> int:
    if self.level >= 17:
      return 2
    if self.level >= 2:
      return 1
    return 0

class Wizard(BaseClassLevel):
  class_type: Literal["Wizard"] = "Wizard"
  spellbook: List[str] = Field(default_factory=list)
  prepared_spells: List[str] = Field(default_factory=list)

In [9]:
DnDClass = Annotated[Union[Fighter, Wizard], Field(discriminator="class_type")]

Here's our character model.

In [10]:
class Character(BaseModel):
  name: str
  base_stats: AbilityScores
  race: DnDRace
  classes: List[DnDClass] = Field(default_factory=list)

  @computed_field
  def total_level(self) -> int:
    return sum(c.level for c in self.classes)
  
  @computed_field
  def proficiency_bonus(self) -> int:
    return ((self.total_level - 1) // 4) + 2
  
  @computed_field
  def character_classes_summary(self) -> str:
    return " / ".join(f"{c.class_type} {c.level}" for c in self.classes)

Let's look at some example usage.

In [ ]:
legolas_data: Dict[str, Union[str, Dict[str, Union[str, int, List[Union[str, Dict]]]]]] = {
  "name": "Legolas",
  "base_stats": {
    "strength": 8,
    "dexterity": 16,
    "constitution": 12,
    "intelligence": 18,
    "wisdom": 10,
    "charisma": 14,
  },
  "race": {
    "race_type": "Elf",
    "darkvision_radius": 60,
    "keen_senses": ["Insight", "Perception", "Survival"],
    "languages": ["Common", "Elvish"],
  },
  "classes": [
    {
      "class_type": "Fighter",
      "level": 2,
      "fighting_style": "Defense",
    },
    {
      "class_type": "Wizard",
      "level": 3,
      "subclass": "School of Evocation",
      "spellbook": ["Mage Armor", "Fireball", "Shield"],
    },
  ],
}

In [12]:
legolas = Character.model_validate(legolas_data)

print(f"Name: {legolas.name}")
print(f"Class: {legolas.character_classes_summary}")
print(f"Proficiency Bonus: +{legolas.proficiency_bonus}")
print(f"Action Surges Available: {legolas.classes[0].action_surges}")

Name: Legolas
Class: Fighter 2 / Wizard 3
Proficiency Bonus: +3
Action Surges Available: 1
